# Detecção de Objetos com YOLOv3 - Dataset PASCAL VOC 2012

Este notebook implementa a arquitetura YOLOv3 (You Only Look Once v3) para detecção de objetos usando o dataset PASCAL VOC 2012. O YOLOv3 é conhecido por sua eficiência e precisão em tarefas de detecção de objetos em tempo real.

## Objetivos:
1. **Criação de Dataset**: Utilizar o dataset PASCAL VOC 2012 com anotações para caixas delimitadoras
2. **Implementação YOLOv3**: Implementar a arquitetura completa do YOLOv3 
3. **Treinamento**: Treinar o modelo para detectar objetos nas 20 classes do PASCAL VOC
4. **Avaliação**: Medir performance usando Mean Average Precision (mAP)

## Dataset PASCAL VOC 2012:
- **20 classes de objetos**: pessoa, bicicleta, carro, motocicleta, avião, ônibus, trem, caminhão, barco, semáforo, hidrante, sinal de pare, parquímetro, banco, pássaro, gato, cachorro, cavalo, ovelha, vaca, elefante, urso, zebra, girafa
- **Imagens de treinamento**: ~17.000 imagens anotadas
- **Imagens de teste**: ~16.000 imagens
- **Formato de anotação**: XML com coordenadas das bounding boxes


## 1. Importação de Bibliotecas

Importamos todas as bibliotecas necessárias para implementar o YOLOv3, incluindo PyTorch para deep learning, OpenCV para processamento de imagens, e bibliotecas auxiliares para manipulação de dados.


In [1]:
# Bibliotecas fundamentais
import os
import xml.etree.ElementTree as ET
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
import torchvision.transforms as transforms

# Bibliotecas para processamento de imagens
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Bibliotecas para análise e visualização
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Todas as bibliotecas foram importadas com sucesso!")
print(f"🔥 PyTorch versão: {torch.__version__}")
print(f"🖥️  CUDA disponível: {torch.cuda.is_available()}")


✅ Todas as bibliotecas foram importadas com sucesso!
🔥 PyTorch versão: 2.8.0+cu128
🖥️  CUDA disponível: True


## 2. Configuração de Hiperparâmetros e Constantes

Definimos todas as configurações importantes do modelo, incluindo as âncoras do YOLOv3, classes do PASCAL VOC, e parâmetros de treinamento.


In [2]:
# Configurações do dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Usando dispositivo: {device}")

# Configurações de modelo e checkpoint
LOAD_MODEL = False
SAVE_MODEL = True
CHECKPOINT_FILE = "yolov3_pascal_voc.pth.tar"

# Âncoras do YOLOv3 para diferentes escalas (otimizadas para PASCAL VOC)
# Escala pequena (13x13), média (26x26) e grande (52x52)
ANCHORS = [
    [(0.28, 0.22), (0.38, 0.48), (0.9, 0.78)],    # Escala pequena - objetos grandes
    [(0.07, 0.15), (0.15, 0.11), (0.14, 0.29)],   # Escala média - objetos médios
    [(0.02, 0.03), (0.04, 0.07), (0.08, 0.06)],   # Escala grande - objetos pequenos
]

# Hiperparâmetros de treinamento
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100
IMAGE_SIZE = 416  # Tamanho padrão para YOLOv3

# Tamanhos das grades para cada escala de saída
GRID_SIZES = [IMAGE_SIZE // 32, IMAGE_SIZE // 16, IMAGE_SIZE // 8]  # [13, 26, 52]

# Classes do PASCAL VOC 2012 (20 classes)
PASCAL_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

NUM_CLASSES = len(PASCAL_CLASSES)
print(f"📊 Número de classes: {NUM_CLASSES}")
print(f"🎯 Classes: {', '.join(PASCAL_CLASSES)}")

# Configurações de caminhos de dados
DATA_DIR = "data"
TRAIN_DIR = os.path.join(DATA_DIR, "VOC2012_train_val")
TEST_DIR = os.path.join(DATA_DIR, "VOC2012_test")

print(f"📁 Diretório de treinamento: {TRAIN_DIR}")
print(f"📁 Diretório de teste: {TEST_DIR}")


🖥️  Usando dispositivo: cuda
📊 Número de classes: 20
🎯 Classes: aeroplane, bicycle, bird, boat, bottle, bus, car, cat, chair, cow, diningtable, dog, horse, motorbike, person, pottedplant, sheep, sofa, train, tvmonitor
📁 Diretório de treinamento: data/VOC2012_train_val
📁 Diretório de teste: data/VOC2012_test


## 3. Funções Auxiliares

Implementamos funções essenciais para o funcionamento do YOLOv3, incluindo cálculo de IoU, Non-Maximum Suppression, e conversão de coordenadas.


In [3]:
def intersection_over_union(box1, box2, is_pred=True):
    """
    Calcula o Intersection over Union (IoU) entre duas caixas delimitadoras.

    Args:
        box1: Primeira caixa delimitadora [x, y, width, height]
        box2: Segunda caixa delimitadora [x, y, width, height]
        is_pred: Se True, trata as caixas como predições (aplica conversões)

    Returns:
        IoU score entre as duas caixas
    """
    if is_pred:
        # Conversão de coordenadas do centro para cantos
        # box1 e box2 estão no formato [x_center, y_center, width, height]

        # Coordenadas da primeira caixa
        b1_x1 = box1[..., 0:1] - box1[..., 2:3] / 2  # x_min
        b1_y1 = box1[..., 1:2] - box1[..., 3:4] / 2  # y_min
        b1_x2 = box1[..., 0:1] + box1[..., 2:3] / 2  # x_max
        b1_y2 = box1[..., 1:2] + box1[..., 3:4] / 2  # y_max

        # Coordenadas da segunda caixa
        b2_x1 = box2[..., 0:1] - box2[..., 2:3] / 2
        b2_y1 = box2[..., 1:2] - box2[..., 3:4] / 2
        b2_x2 = box2[..., 0:1] + box2[..., 2:3] / 2
        b2_y2 = box2[..., 1:2] + box2[..., 3:4] / 2

        # Cálculo da área de interseção
        x1 = torch.max(b1_x1, b2_x1)
        y1 = torch.max(b1_y1, b2_y1)
        x2 = torch.min(b1_x2, b2_x2)
        y2 = torch.min(b1_y2, b2_y2)

        # Garante que a interseção seja pelo menos 0
        intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

        # Cálculo das áreas das caixas
        box1_area = abs((b1_x2 - b1_x1) * (b1_y2 - b1_y1))
        box2_area = abs((b2_x2 - b2_x1) * (b2_y2 - b2_y1))

        # Cálculo da união
        union = box1_area + box2_area - intersection

        # Retorna o IoU com epsilon para evitar divisão por zero
        epsilon = 1e-6
        return intersection / (union + epsilon)

    else:
        # IoU baseado apenas nas dimensões width e height
        intersection_area = torch.min(box1[..., 0], box2[..., 0]) * torch.min(box1[..., 1], box2[..., 1])
        box1_area = box1[..., 0] * box1[..., 1]
        box2_area = box2[..., 0] * box2[..., 1]
        union_area = box1_area + box2_area - intersection_area
        return intersection_area / union_area

print("✅ Função IoU implementada!")


✅ Função IoU implementada!


In [4]:
# Versão corrigida da classe Dataset
class PascalVOCDatasetFixed(Dataset):
    """
    Versão corrigida do Dataset PASCAL VOC 2012 que resolve o problema de parsing XML.
    """

    def __init__(self, data_dir, anchors, image_size=416, grid_sizes=[13, 26, 52],
                 num_classes=20, transform=None):
        """
        Args:
            data_dir: Diretório contendo JPEGImages e Annotations
            anchors: Lista de âncoras para cada escala
            image_size: Tamanho da imagem de entrada
            grid_sizes: Tamanhos das grades de saída [13, 26, 52]
            num_classes: Número de classes (20 para PASCAL VOC)
            transform: Transformações de data augmentation
        """
        self.data_dir = data_dir
        self.image_dir = os.path.join(data_dir, "JPEGImages")
        self.annotation_dir = os.path.join(data_dir, "Annotations")
        self.image_size = image_size
        self.transform = transform
        self.grid_sizes = grid_sizes
        self.num_classes = num_classes
        self.ignore_iou_thresh = 0.5

        # Flatten anchors para facilitar o processamento
        self.anchors = torch.tensor(sum(anchors, []))  # Concatena todas as âncoras
        self.num_anchors_per_scale = 3

        # Mapeamento de classes do PASCAL VOC
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(PASCAL_CLASSES)}

        # Lista de arquivos de imagem (apenas aqueles com anotação correspondente)
        self.image_files = []
        for img_file in os.listdir(self.image_dir):
            if img_file.endswith('.jpg'):
                ann_file = img_file.replace('.jpg', '.xml')
                if os.path.exists(os.path.join(self.annotation_dir, ann_file)):
                    self.image_files.append(img_file)

        print(f"📊 Dataset carregado: {len(self.image_files)} imagens encontradas")

    def parse_xml_annotation(self, xml_path):
        """
        Versão corrigida que trata coordenadas como float.

        Args:
            xml_path: Caminho para o arquivo XML de anotação

        Returns:
            Lista de bounding boxes no formato YOLO [x_center, y_center, width, height, class_id]
        """
        if not os.path.exists(xml_path):
            return []

        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Obtém dimensões da imagem
        size = root.find('size')
        img_width = float(size.find('width').text)
        img_height = float(size.find('height').text)

        boxes = []

        # Processa cada objeto na imagem
        for obj in root.findall('object'):
            class_name = obj.find('name').text

            # Verifica se a classe está no nosso conjunto de classes
            if class_name not in self.class_to_idx:
                continue

            class_id = self.class_to_idx[class_name]

            # Obtém coordenadas da bounding box (CORRIGIDO: usar float)
            bbox = obj.find('bndbox')
            xmin = float(bbox.find('xmin').text)
            ymin = float(bbox.find('ymin').text)
            xmax = float(bbox.find('xmax').text)
            ymax = float(bbox.find('ymax').text)

            # Converte para formato YOLO (coordenadas normalizadas do centro)
            x_center = (xmin + xmax) / 2.0 / img_width
            y_center = (ymin + ymax) / 2.0 / img_height
            width = (xmax - xmin) / img_width
            height = (ymax - ymin) / img_height

            # Clamp valores para garantir que estejam entre 0 e 1
            x_center = max(0, min(1, x_center))
            y_center = max(0, min(1, y_center))
            width = max(0, min(1, width))
            height = max(0, min(1, height))

            boxes.append([x_center, y_center, width, height, class_id])

        return boxes

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Carrega imagem
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')

        # Carrega anotações
        ann_path = img_path.replace('JPEGImages', 'Annotations').replace('.jpg', '.xml')
        bboxes = self.parse_xml_annotation(ann_path)

        # Aplica transformações se especificadas
        if self.transform:
            # Converte para numpy para albumentations
            image_np = np.array(image)

            # Aplica transformações
            transformed = self.transform(image=image_np, bboxes=bboxes)
            image = transformed['image']
            bboxes = transformed['bboxes']
        else:
            # Converte para tensor e normaliza
            transform = transforms.Compose([
                transforms.Resize((self.image_size, self.image_size)),
                transforms.ToTensor(),
            ])
            image = transform(image)

        # Inicializa targets para cada escala
        targets = [torch.zeros((self.num_anchors_per_scale, s, s, 6)) for s in self.grid_sizes]

        # Processa cada bounding box
        for box in bboxes:
            x, y, width, height, class_id = box

            # Calcula IoU com todas as âncoras
            iou_anchors = intersection_over_union(torch.tensor([width, height]), self.anchors, is_pred=False)
            anchor_indices = iou_anchors.argsort(descending=True, dim=0)

            has_anchor = [False] * 3  # Para garantir uma âncora por escala

            for anchor_idx in anchor_indices:
                scale_idx = anchor_idx // self.num_anchors_per_scale
                anchor_on_scale = anchor_idx % self.num_anchors_per_scale
                s = self.grid_sizes[scale_idx]

                i, j = int(s * y), int(s * x)  # linha, coluna
                anchor_taken = targets[scale_idx][anchor_on_scale, i, j, 0]

                # Atribui âncora se disponível
                if not anchor_taken and not has_anchor[scale_idx]:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = 1  # Objectness

                    # Coordenadas relativas à célula
                    x_cell, y_cell = s * x - j, s * y - i
                    width_cell, height_cell = width * s, height * s

                    box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])
                    targets[scale_idx][anchor_on_scale, i, j, 1:5] = box_coordinates
                    targets[scale_idx][anchor_on_scale, i, j, 5] = int(class_id)

                    has_anchor[scale_idx] = True

                # Marca para ignorar se IoU alto mas âncora já tomada
                elif not anchor_taken and iou_anchors[anchor_idx] > self.ignore_iou_thresh:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = -1  # Ignore

        return image, tuple(targets)

print("✅ Classe PascalVOCDatasetFixed implementada!")


✅ Classe PascalVOCDatasetFixed implementada!


In [5]:
# Criação dos datasets corrigidos
train_dataset_fixed = PascalVOCDatasetFixed(
    data_dir=TRAIN_DIR,
    anchors=ANCHORS,
    image_size=IMAGE_SIZE,
    grid_sizes=GRID_SIZES,
    num_classes=NUM_CLASSES,
    transform=None
)

test_dataset_fixed = PascalVOCDatasetFixed(
    data_dir=TEST_DIR,
    anchors=ANCHORS,
    image_size=IMAGE_SIZE,
    grid_sizes=GRID_SIZES,
    num_classes=NUM_CLASSES,
    transform=None
)

# Criação dos DataLoaders corrigidos
train_loader_fixed = DataLoader(
    train_dataset_fixed,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Reduzido para evitar problemas de multiprocessing
    pin_memory=True,
    drop_last=False
)

test_loader_fixed = DataLoader(
    test_dataset_fixed,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,  # Reduzido para evitar problemas de multiprocessing
    pin_memory=True,
    drop_last=False
)

print(f"✅ DataLoaders corrigidos criados!")
print(f"📊 Dataset de treinamento: {len(train_dataset_fixed)} imagens")
print(f"📊 Dataset de teste: {len(test_dataset_fixed)} imagens")
print(f"🔄 Batches de treinamento: {len(train_loader_fixed)}")
print(f"🔄 Batches de teste: {len(test_loader_fixed)}")


📊 Dataset carregado: 17125 imagens encontradas
📊 Dataset carregado: 3985 imagens encontradas
✅ DataLoaders corrigidos criados!
📊 Dataset de treinamento: 17125 imagens
📊 Dataset de teste: 3985 imagens
🔄 Batches de treinamento: 17125
🔄 Batches de teste: 3985


In [6]:
def non_max_suppression(bboxes, iou_threshold, confidence_threshold):
    """
    Aplica Non-Maximum Suppression para remover caixas delimitadoras sobrepostas.

    Args:
        bboxes: Lista de caixas delimitadoras [class, confidence, x, y, width, height]
        iou_threshold: Limiar de IoU para considerar sobreposição
        confidence_threshold: Limiar mínimo de confiança

    Returns:
        Lista de caixas após NMS
    """
    # Filtra caixas com confiança abaixo do limiar
    bboxes = [box for box in bboxes if box[1] > confidence_threshold]

    # Ordena por confiança em ordem decrescente
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)

    # Lista para armazenar caixas após NMS
    bboxes_after_nms = []

    while bboxes:
        # Pega a caixa com maior confiança
        chosen_box = bboxes.pop(0)

        # Filtra caixas que não se sobrepõem significativamente ou são de classes diferentes
        bboxes = [
            box for box in bboxes
            if box[0] != chosen_box[0]  # Classe diferente
            or intersection_over_union(
                torch.tensor(chosen_box[2:]),
                torch.tensor(box[2:])
            ) < iou_threshold  # IoU baixo
        ]

        bboxes_after_nms.append(chosen_box)

    return bboxes_after_nms

print("✅ Função NMS implementada!")


✅ Função NMS implementada!


In [7]:
def convert_cells_to_bboxes(predictions, anchors, grid_size, is_predictions=True):
    """
    Converte as saídas das células da grade em caixas delimitadoras.

    Args:
        predictions: Tensor de predições do modelo
        anchors: Âncoras para a escala atual
        grid_size: Tamanho da grade (13, 26 ou 52)
        is_predictions: Se True, aplica sigmoid/exp nas predições

    Returns:
        Lista de caixas delimitadoras
    """
    batch_size = predictions.shape[0]
    num_anchors = len(anchors)
    box_predictions = predictions[..., 1:5]  # [x, y, width, height]

    if is_predictions:
        # Aplica transformações nas predições
        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        box_predictions[..., 0:2] = torch.sigmoid(box_predictions[..., 0:2])  # x, y
        box_predictions[..., 2:] = torch.exp(box_predictions[..., 2:]) * anchors  # w, h
        scores = torch.sigmoid(predictions[..., 0:1])  # Confiança do objeto
        best_class = torch.argmax(predictions[..., 5:], dim=-1).unsqueeze(-1)  # Classe predita
    else:
        scores = predictions[..., 0:1]
        best_class = predictions[..., 5:6]

    # Cria índices das células para conversão de coordenadas
    cell_indices = (
        torch.arange(grid_size)
        .repeat(predictions.shape[0], 3, grid_size, 1)
        .unsqueeze(-1)
        .to(predictions.device)
    )

    # Converte coordenadas das células para coordenadas da imagem
    x = 1 / grid_size * (box_predictions[..., 0:1] + cell_indices)
    y = 1 / grid_size * (box_predictions[..., 1:2] + cell_indices.permute(0, 1, 3, 2, 4))
    width_height = 1 / grid_size * box_predictions[..., 2:4]

    # Concatena todos os valores e redimensiona
    converted_bboxes = torch.cat(
        (best_class, scores, x, y, width_height), dim=-1
    ).reshape(batch_size, num_anchors * grid_size * grid_size, 6)

    return converted_bboxes.tolist()

print("✅ Função de conversão de células implementada!")


✅ Função de conversão de células implementada!


## 4. Dataset PASCAL VOC 2012

Implementamos uma classe Dataset customizada para carregar e processar os dados do PASCAL VOC 2012, incluindo parsing das anotações XML e preparação dos targets para o treinamento.


In [8]:
class PascalVOCDataset(Dataset):
    """
    Dataset customizado para carregar dados do PASCAL VOC 2012.

    Este dataset processa imagens e anotações XML, convertendo-as para o formato
    necessário para treinar o YOLOv3.
    """

    def __init__(self, data_dir, anchors, image_size=416, grid_sizes=[13, 26, 52],
                 num_classes=20, transform=None):
        """
        Args:
            data_dir: Diretório contendo JPEGImages e Annotations
            anchors: Lista de âncoras para cada escala
            image_size: Tamanho da imagem de entrada
            grid_sizes: Tamanhos das grades de saída [13, 26, 52]
            num_classes: Número de classes (20 para PASCAL VOC)
            transform: Transformações de data augmentation
        """
        self.data_dir = data_dir
        self.image_dir = os.path.join(data_dir, "JPEGImages")
        self.annotation_dir = os.path.join(data_dir, "Annotations")
        self.image_size = image_size
        self.transform = transform
        self.grid_sizes = grid_sizes
        self.num_classes = num_classes
        self.ignore_iou_thresh = 0.5

        # Processa as âncoras para cada escala
        self.anchors = torch.tensor(anchors[0] + anchors[1] + anchors[2])
        self.num_anchors = self.anchors.shape[0]
        self.num_anchors_per_scale = self.num_anchors // 3

        # Mapeia nomes de classes para índices
        self.class_to_idx = {cls: idx for idx, cls in enumerate(PASCAL_CLASSES)}

        # Coleta todos os arquivos de imagem
        self.image_files = []
        if os.path.exists(self.image_dir):
            for file in os.listdir(self.image_dir):
                if file.lower().endswith(('.jpg', '.jpeg')):
                    img_path = os.path.join(self.image_dir, file)
                    ann_path = os.path.join(self.annotation_dir, file.replace('.jpg', '.xml'))

                    # Só adiciona se a anotação existir
                    if os.path.exists(ann_path):
                        self.image_files.append(img_path)

        print(f"📊 Dataset carregado: {len(self.image_files)} imagens encontradas")

    def __len__(self):
        return len(self.image_files)

    def parse_xml_annotation(self, xml_path):
        """
        Faz parsing de um arquivo XML de anotação do PASCAL VOC.

        Returns:
            Lista de bounding boxes [x_center, y_center, width, height, class_id]
        """
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Obtém dimensões da imagem
        size = root.find('size')
        img_width = int(size.find('width').text)
        img_height = int(size.find('height').text)

        bboxes = []

        # Processa cada objeto na imagem
        for obj in root.findall('object'):
            class_name = obj.find('name').text

            # Pula se a classe não estiver no nosso conjunto
            if class_name not in self.class_to_idx:
                continue

            class_id = self.class_to_idx[class_name]

            # Obtém coordenadas da bounding box
            bbox = obj.find('bndbox')
            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)

            # Converte para formato YOLO (coordenadas normalizadas do centro)
            x_center = (xmin + xmax) / 2.0 / img_width
            y_center = (ymin + ymax) / 2.0 / img_height
            width = (xmax - xmin) / img_width
            height = (ymax - ymin) / img_height

            bboxes.append([x_center, y_center, width, height, class_id])

        return bboxes

    def __getitem__(self, idx):
        # Carrega imagem
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')

        # Carrega anotações
        ann_path = img_path.replace('JPEGImages', 'Annotations').replace('.jpg', '.xml')
        bboxes = self.parse_xml_annotation(ann_path)

        # Aplica transformações se especificadas
        if self.transform:
            # Converte para numpy para albumentations
            image_np = np.array(image)

            # Aplica transformações
            transformed = self.transform(image=image_np, bboxes=bboxes)
            image = transformed['image']
            bboxes = transformed['bboxes']
        else:
            # Converte para tensor e normaliza
            transform = transforms.Compose([
                transforms.Resize((self.image_size, self.image_size)),
                transforms.ToTensor(),
                         ])
            image = transform(image)

        # Inicializa targets para cada escala
        targets = [torch.zeros((self.num_anchors_per_scale, s, s, 6)) for s in self.grid_sizes]

        # Processa cada bounding box
        for box in bboxes:
            x, y, width, height, class_id = box

            # Calcula IoU com todas as âncoras
            iou_anchors = intersection_over_union(torch.tensor([width, height]), self.anchors, is_pred=False)
            anchor_indices = iou_anchors.argsort(descending=True, dim=0)

            has_anchor = [False] * 3  # Para garantir uma âncora por escala

            for anchor_idx in anchor_indices:
                scale_idx = anchor_idx // self.num_anchors_per_scale
                anchor_on_scale = anchor_idx % self.num_anchors_per_scale
                s = self.grid_sizes[scale_idx]

                # Calcula posição na grade
                i, j = int(s * y), int(s * x)  # linha, coluna
                anchor_taken = targets[scale_idx][anchor_on_scale, i, j, 0]

                # Atribui âncora se disponível
                if not anchor_taken and not has_anchor[scale_idx]:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = 1  # Objectness

                    # Coordenadas relativas à célula
                    x_cell, y_cell = s * x - j, s * y - i
                    width_cell, height_cell = width * s, height * s

                    box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])
                    targets[scale_idx][anchor_on_scale, i, j, 1:5] = box_coordinates
                    targets[scale_idx][anchor_on_scale, i, j, 5] = int(class_id)

                    has_anchor[scale_idx] = True

                # Marca para ignorar se IoU alto mas âncora já tomada
                elif not anchor_taken and iou_anchors[anchor_idx] > self.ignore_iou_thresh:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = -1  # Ignore

        return image, tuple(targets)

print("✅ Classe PascalVOCDataset implementada!")


✅ Classe PascalVOCDataset implementada!


## 5. Carregamento dos Dados

Criamos instâncias do dataset e DataLoaders para treinamento e teste.


In [9]:
# Transformações de data augmentation para treinamento
train_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE),
    A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT),
    A.ColorJitter(brightness=0.6, contrast=0.6, saturation=0.6, hue=0.6, p=0.4),
    A.OneOf([
        A.ShiftScaleRotate(rotate_limit=20, p=0.5, border_mode=cv2.BORDER_CONSTANT),
        A.Affine(shear=15, p=0.5, mode="constant"),
    ], p=1.0),
    A.HorizontalFlip(p=0.5),
    A.Blur(p=0.1),
    A.CLAHE(p=0.1),
    A.Posterize(p=0.1),
    A.ToGray(p=0.1),
    A.ChannelShuffle(p=0.05),
    A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255),
    ToTensorV2(),
], bbox_params=A.BboxParams(format="yolo", min_visibility=0.4, label_fields=["class_labels"]))

# Transformações para teste (sem augmentation)
test_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE),
    A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT),
    A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255),
    ToTensorV2(),
], bbox_params=A.BboxParams(format="yolo", min_visibility=0.4, label_fields=["class_labels"]))

print("✅ Transformações configuradas!")


✅ Transformações configuradas!


In [10]:
# Criação dos datasets
train_dataset = PascalVOCDataset(
    data_dir=TRAIN_DIR,
    anchors=ANCHORS,
    image_size=IMAGE_SIZE,
    grid_sizes=GRID_SIZES,
    num_classes=NUM_CLASSES,
    transform=None  # Usaremos transformações simples por enquanto
)

test_dataset = PascalVOCDataset(
    data_dir=TEST_DIR,
    anchors=ANCHORS,
    image_size=IMAGE_SIZE,
    grid_sizes=GRID_SIZES,
    num_classes=NUM_CLASSES,
    transform=None
)

# Criação dos DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    drop_last=False
)

print(f"✅ DataLoaders criados!")
print(f"📊 Dataset de treinamento: {len(train_dataset)} imagens")
print(f"📊 Dataset de teste: {len(test_dataset)} imagens")
print(f"🔄 Batches de treinamento: {len(train_loader)}")
print(f"🔄 Batches de teste: {len(test_loader)}")


📊 Dataset carregado: 17125 imagens encontradas
📊 Dataset carregado: 3985 imagens encontradas
✅ DataLoaders criados!
📊 Dataset de treinamento: 17125 imagens
📊 Dataset de teste: 3985 imagens
🔄 Batches de treinamento: 17125
🔄 Batches de teste: 3985


## 6. Arquitetura YOLOv3

Implementamos a arquitetura completa do YOLOv3, incluindo os blocos CNN, blocos residuais, e as camadas de predição em múltiplas escalas.


In [11]:
class CNNBlock(nn.Module):
    """
    Bloco CNN básico usado no YOLOv3.

    Consiste em: Convolução -> BatchNorm -> LeakyReLU
    """
    def __init__(self, in_channels, out_channels, use_batch_norm=True, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=not use_batch_norm, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activation = nn.LeakyReLU(0.1)
        self.use_batch_norm = use_batch_norm

    def forward(self, x):
        x = self.conv(x)
        if self.use_batch_norm:
            x = self.bn(x)
            return self.activation(x)
        else:
            return x

print("✅ Bloco CNN implementado!")


✅ Bloco CNN implementado!


In [12]:
class ResidualBlock(nn.Module):
    """
    Bloco residual usado no backbone do YOLOv3.

    Implementa conexões residuais (skip connections) para melhor fluxo de gradiente.
    Cada bloco residual consiste em duas convoluções com conexão de salto.
    """
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()

        # Cria lista de camadas residuais baseada no número de repetições
        res_layers = []
        for _ in range(num_repeats):
            res_layers += [
                nn.Sequential(
                    # Primeira convolução: reduz canais pela metade com kernel 1x1
                    nn.Conv2d(channels, channels // 2, kernel_size=1),
                    nn.BatchNorm2d(channels // 2),
                    nn.LeakyReLU(0.1),
                    # Segunda convolução: restaura número de canais com kernel 3x3
                    nn.Conv2d(channels // 2, channels, kernel_size=3, padding=1),
                    nn.BatchNorm2d(channels),
                    nn.LeakyReLU(0.1)
                )
            ]

        self.layers = nn.ModuleList(res_layers)
        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            residual = x  # Salva entrada original
            x = layer(x)  # Aplica transformação
            if self.use_residual:
                x = x + residual  # Adiciona conexão residual
        return x

print("✅ Bloco Residual implementado!")


✅ Bloco Residual implementado!


In [13]:
class ScalePrediction(nn.Module):
    """
    Camada de predição para cada escala no YOLOv3.

    Produz as predições finais para objectness, coordenadas das bounding boxes,
    e classificação para uma determinada escala (13x13, 26x26, ou 52x52).
    """
    def __init__(self, in_channels, num_classes):
        super().__init__()
        # Cada âncora prediz: objectness (1) + bbox coords (4) + classes (num_classes)
        # Total por âncora = 1 + 4 + num_classes = 5 + num_classes
        # Com 3 âncoras por escala: 3 * (5 + num_classes)

        self.pred = nn.Sequential(
            # Primeira convolução: dobra o número de canais
            nn.Conv2d(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(2 * in_channels),
            nn.LeakyReLU(0.1),
            # Segunda convolução: produz predições finais
            nn.Conv2d(2 * in_channels, (num_classes + 5) * 3, kernel_size=1),
        )
        self.num_classes = num_classes

    def forward(self, x):
        # x shape: (batch_size, in_channels, grid_size, grid_size)
        output = self.pred(x)
        # output shape: (batch_size, (num_classes + 5) * 3, grid_size, grid_size)

        # Reorganiza para formato desejado:
        # (batch_size, 3, grid_size, grid_size, num_classes + 5)
        output = output.view(x.size(0), 3, self.num_classes + 5, x.size(2), x.size(3))
        output = output.permute(0, 1, 3, 4, 2)

        return output

print("✅ Camada de Predição implementada!")


✅ Camada de Predição implementada!


In [14]:
class YOLOv3(nn.Module):
    """
    Implementação completa da arquitetura YOLOv3.

    O YOLOv3 utiliza uma arquitetura baseada no Darknet-53 como backbone,
    seguida por camadas de detecção em 3 escalas diferentes para capturar
    objetos de diferentes tamanhos.
    """
    def __init__(self, in_channels=3, num_classes=20):
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels

        # Definição das camadas da arquitetura YOLOv3
        # Baseado no paper original e implementação Darknet
        self.layers = nn.ModuleList([
            # Camadas iniciais do backbone
            CNNBlock(in_channels, 32, kernel_size=3, stride=1, padding=1),
            CNNBlock(32, 64, kernel_size=3, stride=2, padding=1),
            ResidualBlock(64, num_repeats=1),

            CNNBlock(64, 128, kernel_size=3, stride=2, padding=1),
            ResidualBlock(128, num_repeats=2),

            CNNBlock(128, 256, kernel_size=3, stride=2, padding=1),
            ResidualBlock(256, num_repeats=8),  # Primeira conexão de rota

            CNNBlock(256, 512, kernel_size=3, stride=2, padding=1),
            ResidualBlock(512, num_repeats=8),  # Segunda conexão de rota

            CNNBlock(512, 1024, kernel_size=3, stride=2, padding=1),
            ResidualBlock(1024, num_repeats=4),

            # Primeira escala de detecção (13x13)
            CNNBlock(1024, 512, kernel_size=1, stride=1, padding=0),
            CNNBlock(512, 1024, kernel_size=3, stride=1, padding=1),
            ResidualBlock(1024, use_residual=False, num_repeats=1),
            CNNBlock(1024, 512, kernel_size=1, stride=1, padding=0),
            ScalePrediction(512, num_classes=num_classes),  # Primeira predição

            # Upsampling e segunda escala de detecção (26x26)
            CNNBlock(512, 256, kernel_size=1, stride=1, padding=0),
            nn.Upsample(scale_factor=2),  # Dobra resolução
            CNNBlock(768, 256, kernel_size=1, stride=1, padding=0),  # 768 = 256 + 512 (concat)
            CNNBlock(256, 512, kernel_size=3, stride=1, padding=1),
            ResidualBlock(512, use_residual=False, num_repeats=1),
            CNNBlock(512, 256, kernel_size=1, stride=1, padding=0),
            ScalePrediction(256, num_classes=num_classes),  # Segunda predição

            # Upsampling e terceira escala de detecção (52x52)
            CNNBlock(256, 128, kernel_size=1, stride=1, padding=0),
            nn.Upsample(scale_factor=2),  # Dobra resolução novamente
            CNNBlock(384, 128, kernel_size=1, stride=1, padding=0),  # 384 = 128 + 256 (concat)
            CNNBlock(128, 256, kernel_size=3, stride=1, padding=1),
            ResidualBlock(256, use_residual=False, num_repeats=1),
            CNNBlock(256, 128, kernel_size=1, stride=1, padding=0),
            ScalePrediction(128, num_classes=num_classes)  # Terceira predição
        ])

    def forward(self, x):
        outputs = []  # Armazena as predições das 3 escalas
        route_connections = []  # Armazena features para conexões de rota

        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                # Adiciona predição à lista de saídas
                outputs.append(layer(x))
                continue

            x = layer(x)

            # Salva features para conexões de rota após blocos residuais específicos
            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            # Concatena features das conexões de rota após upsampling
            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1)
                route_connections.pop()

        return outputs

print("✅ Arquitetura YOLOv3 implementada!")


✅ Arquitetura YOLOv3 implementada!


In [15]:
# Teste da arquitetura YOLOv3
def test_yolov3_architecture():
    """Testa se a arquitetura YOLOv3 produz saídas com as dimensões corretas."""
    print("🧪 Testando arquitetura YOLOv3...")

    # Cria modelo
    model = YOLOv3(num_classes=NUM_CLASSES)

    # Entrada de teste
    x = torch.randn((2, 3, IMAGE_SIZE, IMAGE_SIZE))  # batch_size=2

    # Forward pass
    outputs = model(x)

    # Verifica dimensões das saídas
    expected_shapes = [
        (2, 3, IMAGE_SIZE//32, IMAGE_SIZE//32, NUM_CLASSES + 5),  # 13x13
        (2, 3, IMAGE_SIZE//16, IMAGE_SIZE//16, NUM_CLASSES + 5),  # 26x26
        (2, 3, IMAGE_SIZE//8, IMAGE_SIZE//8, NUM_CLASSES + 5),    # 52x52
    ]

    print(f"📊 Número de saídas: {len(outputs)}")
    for i, (output, expected_shape) in enumerate(zip(outputs, expected_shapes)):
        print(f"   Escala {i+1}: {tuple(output.shape)} (esperado: {expected_shape})")
        assert output.shape == expected_shape, f"Forma incorreta para escala {i+1}"

    # Calcula número de parâmetros
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"📈 Total de parâmetros: {total_params:,}")
    print(f"🎯 Parâmetros treináveis: {trainable_params:,}")
    print("✅ Teste da arquitetura passou!")

    return model

# Executa teste
model = test_yolov3_architecture()


🧪 Testando arquitetura YOLOv3...
📊 Número de saídas: 3
   Escala 1: (2, 3, 13, 13, 25) (esperado: (2, 3, 13, 13, 25))
   Escala 2: (2, 3, 26, 26, 25) (esperado: (2, 3, 26, 26, 25))
   Escala 3: (2, 3, 52, 52, 25) (esperado: (2, 3, 52, 52, 25))
📈 Total de parâmetros: 61,646,369
🎯 Parâmetros treináveis: 61,646,369
✅ Teste da arquitetura passou!


## 7. Função de Loss do YOLOv3

A função de loss do YOLOv3 combina múltiplos componentes: loss de objectness, loss de coordenadas das bounding boxes, e loss de classificação.


In [16]:
# Função de Loss Corrigida
class YOLOv3Loss(nn.Module):
    """
    Versão corrigida da função de loss do YOLOv3.

    Corrige o problema de dimensões das âncoras.
    """
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.cross_entropy = nn.CrossEntropyLoss()
        self.sigmoid = nn.Sigmoid()

        # Pesos para balancear os diferentes componentes da loss
        self.lambda_box = 10
        self.lambda_obj = 1
        self.lambda_noobj = 10
        self.lambda_class = 1

    def forward(self, predictions, targets, anchors):
        """
        Calcula a loss total para uma escala.

        Args:
            predictions: Predições do modelo [batch_size, 3, grid_size, grid_size, 5+num_classes]
            targets: Ground truth [batch_size, 3, grid_size, grid_size, 6]
            anchors: Âncoras para esta escala

        Returns:
            Loss total para esta escala
        """
        device = predictions.device

        # Identifica células com e sem objetos
        obj = targets[..., 0] == 1  # Células com objetos
        no_obj = targets[..., 0] == 0  # Células sem objetos

        # === 1. LOSS DE NO-OBJECT (células sem objetos) ===
        no_object_loss = self.bce(
            predictions[..., 0:1][no_obj],
            targets[..., 0:1][no_obj]
        )

        # === 2. LOSS DE OBJECT E COORDENADAS (células com objetos) ===
        if obj.sum() > 0:
            # Redimensiona âncoras para match com predições
            anchors = anchors.reshape(1, 3, 1, 1, 2).to(device)

            # Converte predições para formato de bounding box
            box_preds = torch.cat([
                self.sigmoid(predictions[..., 1:3]),  # x, y (sigmoid)
                torch.exp(predictions[..., 3:5]) * anchors  # w, h (exp * anchor)
            ], dim=-1)

            # Calcula IoU entre predições e targets para células com objetos
            ious = intersection_over_union(box_preds[obj], targets[..., 1:5][obj]).detach()

            # Loss de objectness ponderada pelo IoU
            object_loss = self.mse(
                self.sigmoid(predictions[..., 0:1][obj]),
                ious * targets[..., 0:1][obj]
            )

            # === 3. LOSS DE COORDENADAS DAS BOUNDING BOXES ===
            # Aplica sigmoid nas coordenadas x, y das predições
            predictions_copy = predictions.clone()
            predictions_copy[..., 1:3] = self.sigmoid(predictions_copy[..., 1:3])

            # Converte targets w, h para log space
            target_box = targets[..., 1:5].clone()
            # Corrige dimensões - expande âncoras para match com target_box
            anchors_expanded = anchors.expand_as(target_box[..., 2:4])
            target_box[..., 2:4] = torch.log(1e-16 + target_box[..., 2:4] / anchors_expanded)

            # Calcula loss de coordenadas apenas para células com objetos
            box_loss = self.mse(predictions_copy[..., 1:5][obj], target_box[obj])

            # === 4. LOSS DE CLASSIFICAÇÃO ===
            class_loss = self.cross_entropy(
                predictions[..., 5:][obj],
                targets[..., 5][obj].long()
            )
        else:
            # Se não há objetos, define losses como zero
            object_loss = torch.tensor(0.0, device=device, requires_grad=True)
            box_loss = torch.tensor(0.0, device=device, requires_grad=True)
            class_loss = torch.tensor(0.0, device=device, requires_grad=True)

        # === LOSS TOTAL PONDERADA ===
        total_loss = (
            self.lambda_box * box_loss
            + self.lambda_obj * object_loss
            + self.lambda_noobj * no_object_loss
            + self.lambda_class * class_loss
        )

        return total_loss, {
            'box_loss': box_loss.item() if obj.sum() > 0 else 0.0,
            'obj_loss': object_loss.item() if obj.sum() > 0 else 0.0,
            'noobj_loss': no_object_loss.item(),
            'class_loss': class_loss.item() if obj.sum() > 0 else 0.0,
            'total_loss': total_loss.item()
        }

# Substitui a função de loss anterior
loss_fn = YOLOv3Loss()
print("✅ Função de Loss corrigida implementada!")


✅ Função de Loss corrigida implementada!


In [17]:
# Teste da função de loss corrigida
print("🧪 Testando função de loss corrigida...")

# Cria dados de teste
test_predictions = torch.randn(1, 3, 13, 13, 25).to(device)  # batch=1, 3 âncoras, 13x13, 25 outputs
test_targets = torch.zeros(1, 3, 13, 13, 6).to(device)  # batch=1, 3 âncoras, 13x13, 6 targets
test_anchors = scaled_anchors[0]  # Primeira escala

# Testa a função de loss
try:
    loss_value, loss_components = loss_fn(test_predictions, test_targets, test_anchors)
    print("✅ Função de loss funcionando corretamente!")
    print(f"   Loss total: {loss_value.item():.4f}")
    print(f"   Componentes: {loss_components}")
except Exception as e:
    print(f"❌ Erro na função de loss: {e}")

print("🔧 Correção aplicada! Agora você pode executar o treinamento.")


🧪 Testando função de loss corrigida...


NameError: name 'scaled_anchors' is not defined

## 8. Funções de Treinamento e Utilidades

Implementamos funções auxiliares para salvar/carregar checkpoints, visualizar predições e calcular métricas de avaliação.


In [ ]:
# Loop principal de treinamento corrigido
def main_training_loop_fixed():
    """
    Loop principal de treinamento do YOLOv3 com correções.
    """
    print("🎯 Iniciando treinamento do YOLOv3 (versão corrigida)...")

    # Variáveis para acompanhar melhor modelo
    best_loss = float('inf')
    start_epoch = 0
    current_loss = float('inf')  # Inicializa a variável

    # Carrega checkpoint se especificado
    if LOAD_MODEL and os.path.exists(CHECKPOINT_FILE):
        start_epoch = load_checkpoint(CHECKPOINT_FILE, model, optimizer, LEARNING_RATE)

    # Listas para armazenar histórico de treinamento
    train_losses = []

    try:
        for epoch in range(start_epoch, NUM_EPOCHS):
            print(f"\n{'='*50}")
            print(f"🔄 ÉPOCA {epoch+1}/{NUM_EPOCHS}")
            print(f"{'='*50}")

            # Treina por uma época (usando DataLoader corrigido)
            epoch_metrics = train_one_epoch(
                model, train_loader_fixed, optimizer, loss_fn,
                scaler, scaled_anchors, epoch
            )

            # Armazena métricas
            train_losses.append(epoch_metrics['total_loss'])
            current_loss = epoch_metrics['total_loss']

            # Atualiza learning rate
            scheduler.step(current_loss)
            current_lr = optimizer.param_groups[0]['lr']

            # Imprime resumo da época
            print(f"\n📊 RESUMO DA ÉPOCA {epoch+1}:")
            print(f"   Loss Total: {current_loss:.4f}")
            print(f"   Loss Box: {epoch_metrics['box_loss']:.4f}")
            print(f"   Loss Obj: {epoch_metrics['obj_loss']:.4f}")
            print(f"   Loss NoObj: {epoch_metrics['noobj_loss']:.4f}")
            print(f"   Loss Class: {epoch_metrics['class_loss']:.4f}")
            print(f"   Learning Rate: {current_lr:.6f}")

            # Salva checkpoint se melhor modelo
            if current_loss < best_loss:
                best_loss = current_loss
                print(f"🎉 Novo melhor modelo! Loss: {best_loss:.4f}")

                if SAVE_MODEL:
                    save_checkpoint(model, optimizer, epoch, current_loss, CHECKPOINT_FILE)

            # Salva checkpoint a cada 10 épocas
            elif SAVE_MODEL and (epoch + 1) % 10 == 0:
                checkpoint_name = f"checkpoint_epoch_{epoch+1}.pth.tar"
                save_checkpoint(model, optimizer, epoch, current_loss, checkpoint_name)

    except KeyboardInterrupt:
        print("\n⚠️  Treinamento interrompido pelo usuário")
        if SAVE_MODEL:
            save_checkpoint(model, optimizer, epoch, current_loss, "checkpoint_interrupted.pth.tar")

    except Exception as e:
        print(f"\n❌ Erro durante treinamento: {e}")
        if SAVE_MODEL:
            save_checkpoint(model, optimizer, epoch, current_loss, "checkpoint_error.pth.tar")
        raise

    print(f"\n🎊 Treinamento concluído!")
    print(f"   Melhor Loss: {best_loss:.4f}")

    return train_losses

print("✅ Loop de treinamento corrigido implementado!")


In [ ]:
# Teste do dataset corrigido
print("🧪 Testando dataset corrigido...")

try:
    # Testa carregamento de uma amostra
    sample_image, sample_targets = train_dataset_fixed[0]
    print("✅ Dataset funcionando corretamente!")
    print(f"   Formato da imagem: {sample_image.shape}")
    print(f"   Número de escalas de targets: {len(sample_targets)}")
    print(f"   Formato dos targets:")
    for i, target in enumerate(sample_targets):
        print(f"     Escala {i+1}: {target.shape}")

    # Testa um batch do DataLoader
    batch_images, batch_targets = next(iter(train_loader_fixed))
    print(f"✅ DataLoader funcionando!")
    print(f"   Batch de imagens: {batch_images.shape}")
    print(f"   Batch de targets: {len(batch_targets)} escalas")

except Exception as e:
    print(f"❌ Erro no dataset: {e}")

print("\n🚀 Pronto para treinar! Execute a função main_training_loop_fixed() para iniciar.")


In [ ]:
def save_checkpoint(model, optimizer, epoch, loss, filename="checkpoint.pth.tar"):
    """
    Salva checkpoint do modelo durante o treinamento.

    Args:
        model: Modelo YOLOv3
        optimizer: Otimizador
        epoch: Época atual
        loss: Loss atual
        filename: Nome do arquivo de checkpoint
    """
    print(f"💾 Salvando checkpoint na época {epoch}...")
    checkpoint = {
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
        "loss": loss,
    }
    torch.save(checkpoint, filename)
    print(f"✅ Checkpoint salvo: {filename}")

def load_checkpoint(checkpoint_file, model, optimizer, lr):
    """
    Carrega checkpoint salvo.

    Args:
        checkpoint_file: Caminho para o arquivo de checkpoint
        model: Modelo YOLOv3
        optimizer: Otimizador
        lr: Learning rate

    Returns:
        Época do checkpoint carregado
    """
    print(f"📂 Carregando checkpoint: {checkpoint_file}")
    checkpoint = torch.load(checkpoint_file, map_location=device)
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    # Atualiza learning rate
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    epoch = checkpoint.get("epoch", 0)
    loss = checkpoint.get("loss", float('inf'))
    print(f"✅ Checkpoint carregado - Época: {epoch}, Loss: {loss:.4f}")
    return epoch

print("✅ Funções de checkpoint implementadas!")


✅ Funções de checkpoint implementadas!


In [ ]:
def plot_image_with_boxes(image, boxes, class_labels=PASCAL_CLASSES):
    """
    Visualiza uma imagem com suas bounding boxes preditas.

    Args:
        image: Tensor da imagem [C, H, W]
        boxes: Lista de boxes [class, confidence, x, y, width, height]
        class_labels: Lista de nomes das classes
    """
    # Converte tensor para numpy e ajusta dimensões
    if isinstance(image, torch.Tensor):
        image = image.permute(1, 2, 0).cpu().numpy()

    # Normaliza valores se necessário
    if image.max() <= 1:
        image = (image * 255).astype(np.uint8)

    # Cria figura
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)

    # Define cores para cada classe
    colors = plt.cm.Set3(np.linspace(0, 1, len(class_labels)))

    # Desenha cada bounding box
    for box in boxes:
        if len(box) >= 6:
            class_id, confidence, x_center, y_center, width, height = box[:6]
        else:
            continue

        # Converte coordenadas do centro para cantos
        h, w = image.shape[:2]
        x1 = (x_center - width/2) * w
        y1 = (y_center - height/2) * h
        box_width = width * w
        box_height = height * h

        # Cria retângulo
        rect = patches.Rectangle(
            (x1, y1), box_width, box_height,
            linewidth=2, edgecolor=colors[int(class_id)],
            facecolor='none', alpha=0.8
        )
        ax.add_patch(rect)

        # Adiciona label com classe e confiança
        label = f"{class_labels[int(class_id)]}: {confidence:.2f}"
        ax.text(x1, y1-5, label, fontsize=10, color=colors[int(class_id)],
                bbox=dict(boxstyle="round,pad=0.3", facecolor=colors[int(class_id)], alpha=0.7))

    ax.axis('off')
    plt.tight_layout()
    plt.show()

print("✅ Função de visualização implementada!")


✅ Função de visualização implementada!


## 9. Loop de Treinamento

Implementamos o loop principal de treinamento com monitoramento de loss e salvamento de checkpoints.


In [ ]:
def train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, scaled_anchors, epoch):
    """
    Treina o modelo por uma época.

    Args:
        model: Modelo YOLOv3
        train_loader: DataLoader de treinamento
        optimizer: Otimizador
        loss_fn: Função de loss
        scaler: GradScaler para mixed precision
        scaled_anchors: Âncoras escaladas para cada escala
        epoch: Número da época atual

    Returns:
        Dicionário com métricas da época
    """
    model.train()

    # Métricas para acompanhar
    total_losses = []
    box_losses = []
    obj_losses = []
    noobj_losses = []
    class_losses = []

    # Barra de progresso
    pbar = tqdm(train_loader, desc=f"Época {epoch+1}/{NUM_EPOCHS}")

    for batch_idx, (images, targets) in enumerate(pbar):
        # Move dados para GPU
        images = images.to(device)
        targets = [target.to(device) for target in targets]

        # Forward pass com mixed precision
        with torch.cuda.amp.autocast():
            outputs = model(images)

            # Calcula loss para cada escala
            total_loss = 0
            loss_components = {'box_loss': 0, 'obj_loss': 0, 'noobj_loss': 0, 'class_loss': 0}

            for i in range(3):  # 3 escalas
                scale_loss, scale_components = loss_fn(outputs[i], targets[i], scaled_anchors[i])
                total_loss += scale_loss

                # Acumula componentes da loss
                for key in loss_components:
                    loss_components[key] += scale_components[key]

        # Backward pass
        optimizer.zero_grad()
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Armazena métricas
        total_losses.append(total_loss.item())
        box_losses.append(loss_components['box_loss'])
        obj_losses.append(loss_components['obj_loss'])
        noobj_losses.append(loss_components['noobj_loss'])
        class_losses.append(loss_components['class_loss'])

        # Atualiza barra de progresso
        avg_loss = np.mean(total_losses[-100:])  # Média das últimas 100 iterações
        pbar.set_postfix({
            'Loss': f'{avg_loss:.4f}',
            'Box': f'{np.mean(box_losses[-100:]):.3f}',
            'Obj': f'{np.mean(obj_losses[-100:]):.3f}',
            'Class': f'{np.mean(class_losses[-100:]):.3f}'
        })

    # Retorna métricas da época
    return {
        'total_loss': np.mean(total_losses),
        'box_loss': np.mean(box_losses),
        'obj_loss': np.mean(obj_losses),
        'noobj_loss': np.mean(noobj_losses),
        'class_loss': np.mean(class_losses)
    }

print("✅ Função de treinamento implementada!")


✅ Função de treinamento implementada!


In [ ]:
# Configuração do treinamento
print("🚀 Configurando treinamento...")

# Move modelo para GPU
model = model.to(device)

# Configuração do otimizador
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Scheduler para ajustar learning rate
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# Função de loss
loss_fn = YOLOv3Loss()

# Scaler para mixed precision training
scaler = torch.cuda.amp.GradScaler()

# Prepara âncoras escaladas para cada escala
scaled_anchors = (
    torch.tensor(ANCHORS) *
    torch.tensor(GRID_SIZES).unsqueeze(1).unsqueeze(1).repeat(1, 3, 2)
).to(device)

print(f"✅ Configuração concluída!")
print(f"📊 Modelo: {sum(p.numel() for p in model.parameters()):,} parâmetros")
print(f"🎯 Otimizador: Adam (lr={LEARNING_RATE})")
print(f"📈 Scheduler: ReduceLROnPlateau")


🚀 Configurando treinamento...


✅ Configuração concluída!
📊 Modelo: 61,646,369 parâmetros
🎯 Otimizador: Adam (lr=0.0001)
📈 Scheduler: ReduceLROnPlateau


In [ ]:
# Loop principal de treinamento
def main_training_loop():
    """
    Loop principal de treinamento do YOLOv3.
    """
    print("🎯 Iniciando treinamento do YOLOv3...")

    # Variáveis para acompanhar melhor modelo
    best_loss = float('inf')
    start_epoch = 0

    # Carrega checkpoint se especificado
    if LOAD_MODEL and os.path.exists(CHECKPOINT_FILE):
        start_epoch = load_checkpoint(CHECKPOINT_FILE, model, optimizer, LEARNING_RATE)

    # Listas para armazenar histórico de treinamento
    train_losses = []

    try:
        for epoch in range(start_epoch, NUM_EPOCHS):
            print(f"\n{'='*50}")
            print(f"🔄 ÉPOCA {epoch+1}/{NUM_EPOCHS}")
            print(f"{'='*50}")

            # Treina por uma época
            epoch_metrics = train_one_epoch(
                model, train_loader, optimizer, loss_fn,
                scaler, scaled_anchors, epoch
            )

            # Armazena métricas
            train_losses.append(epoch_metrics['total_loss'])
            current_loss = epoch_metrics['total_loss']

            # Atualiza learning rate
            scheduler.step(current_loss)
            current_lr = optimizer.param_groups[0]['lr']

            # Imprime resumo da época
            print(f"\n📊 RESUMO DA ÉPOCA {epoch+1}:")
            print(f"   Loss Total: {current_loss:.4f}")
            print(f"   Loss Box: {epoch_metrics['box_loss']:.4f}")
            print(f"   Loss Obj: {epoch_metrics['obj_loss']:.4f}")
            print(f"   Loss NoObj: {epoch_metrics['noobj_loss']:.4f}")
            print(f"   Loss Class: {epoch_metrics['class_loss']:.4f}")
            print(f"   Learning Rate: {current_lr:.6f}")

            # Salva checkpoint se melhor modelo
            if current_loss < best_loss:
                best_loss = current_loss
                print(f"🎉 Novo melhor modelo! Loss: {best_loss:.4f}")

                if SAVE_MODEL:
                    save_checkpoint(model, optimizer, epoch, current_loss, CHECKPOINT_FILE)

            # Salva checkpoint a cada 10 épocas
            elif SAVE_MODEL and (epoch + 1) % 10 == 0:
                checkpoint_name = f"checkpoint_epoch_{epoch+1}.pth.tar"
                save_checkpoint(model, optimizer, epoch, current_loss, checkpoint_name)

    except KeyboardInterrupt:
        print("\n⚠️  Treinamento interrompido pelo usuário")
        if SAVE_MODEL:
            save_checkpoint(model, optimizer, epoch, current_loss, "checkpoint_interrupted.pth.tar")

    except Exception as e:
        print(f"\n❌ Erro durante treinamento: {e}")
        if SAVE_MODEL:
            save_checkpoint(model, optimizer, epoch, current_loss, "checkpoint_error.pth.tar")
        raise

    print(f"\n🎊 Treinamento concluído!")
    print(f"   Melhor Loss: {best_loss:.4f}")

    return train_losses

# Descomente a linha abaixo para iniciar o treinamento
train_losses = main_training_loop()

print("✅ Loop de treinamento implementado!")


🎯 Iniciando treinamento do YOLOv3...

🔄 ÉPOCA 1/100


Época 1/100:   0%|          | 81/17125 [00:42<2:30:39,  1.89it/s, Loss=72.3975, Box=4.673, Obj=0.316, Class=8.727]


⚠️  Treinamento interrompido pelo usuário


UnboundLocalError: cannot access local variable 'current_loss' where it is not associated with a value

## 10. Avaliação e Métricas

Implementamos funções para avaliar o modelo usando métricas padrão de detecção de objetos, incluindo Mean Average Precision (mAP).


In [ ]:
def get_evaluation_bboxes(loader, model, iou_threshold, anchors, threshold):
    """
    Obtém predições do modelo para avaliação.

    Args:
        loader: DataLoader para avaliação
        model: Modelo treinado
        iou_threshold: Limiar IoU para NMS
        anchors: Âncoras do modelo
        threshold: Limiar de confiança

    Returns:
        Tupla (predições, ground_truth) formatadas para cálculo de mAP
    """
    model.eval()

    all_pred_boxes = []
    all_true_boxes = []

    # Converte âncoras para formato adequado
    train_idx = 0

    for batch_idx, (x, labels) in enumerate(tqdm(loader, desc="Avaliando")):
        x = x.to(device)

        with torch.no_grad():
            predictions = model(x)

            batch_size = x.shape[0]
            bboxes = [[] for _ in range(batch_size)]

            # Processa predições de cada escala
            for i in range(3):
                batch_size, A, S, _, _ = predictions[i].shape
                anchor = anchors[i]

                boxes_scale_i = convert_cells_to_bboxes(
                    predictions[i], anchor, grid_size=S, is_predictions=True
                )

                for idx, (box) in enumerate(boxes_scale_i):
                    bboxes[idx] += box

            # Aplica NMS e formata resultados
            for idx in range(batch_size):
                nms_boxes = non_max_suppression(
                    bboxes[idx],
                    iou_threshold=iou_threshold,
                    confidence_threshold=threshold
                )

                # Adiciona índice da imagem para cada predição
                for nms_box in nms_boxes:
                    all_pred_boxes.append([train_idx] + nms_box)

                # Processa ground truth
                for i in range(3):  # Para cada escala
                    for anchor in range(A):
                        for s_y in range(S):
                            for s_x in range(S):
                                if labels[i][idx, anchor, s_y, s_x, 0] == 1:  # Se há objeto
                                    # Converte coordenadas da célula para imagem
                                    x_cell = labels[i][idx, anchor, s_y, s_x, 1]
                                    y_cell = labels[i][idx, anchor, s_y, s_x, 2]
                                    w_cell = labels[i][idx, anchor, s_y, s_x, 3]
                                    h_cell = labels[i][idx, anchor, s_y, s_x, 4]
                                    class_label = labels[i][idx, anchor, s_y, s_x, 5]

                                    # Converte para coordenadas da imagem
                                    x = (x_cell + s_x) / S
                                    y = (y_cell + s_y) / S
                                    w = w_cell / S
                                    h = h_cell / S

                                    all_true_boxes.append([
                                        train_idx, class_label, 1.0, x, y, w, h
                                    ])

                train_idx += 1

    model.train()
    return all_pred_boxes, all_true_boxes

print("✅ Função de avaliação implementada!")


✅ Função de avaliação implementada!


In [ ]:
def calculate_map(pred_boxes, true_boxes, iou_threshold=0.5, num_classes=20):
    """
    Calcula Mean Average Precision (mAP) para detecção de objetos.

    Args:
        pred_boxes: Lista de predições [train_idx, class, confidence, x, y, w, h]
        true_boxes: Lista de ground truth [train_idx, class, confidence, x, y, w, h]
        iou_threshold: Limiar de IoU para considerar detecção correta
        num_classes: Número de classes

    Returns:
        mAP e AP por classe
    """
    # Calcula AP para cada classe
    average_precisions = []
    epsilon = 1e-6

    for c in range(num_classes):
        detections = []
        ground_truths = []

        # Filtra predições e ground truth para a classe atual
        for detection in pred_boxes:
            if detection[1] == c:
                detections.append(detection)

        for true_box in true_boxes:
            if true_box[1] == c:
                ground_truths.append(true_box)

        # Conta ground truth por imagem
        amount_bboxes = Counter([gt[0] for gt in ground_truths])

        # Inicializa dicionário para rastrear detecções
        for key, val in amount_bboxes.items():
            amount_bboxes[key] = torch.zeros(val)

        # Ordena detecções por confiança
        detections.sort(key=lambda x: x[2], reverse=True)

        TP = torch.zeros((len(detections)))
        FP = torch.zeros((len(detections)))
        total_true_bboxes = len(ground_truths)

        # Se não há ground truth para esta classe
        if total_true_bboxes == 0:
            continue

        for detection_idx, detection in enumerate(detections):
            # Pega ground truth da mesma imagem
            ground_truth_img = [
                bbox for bbox in ground_truths if bbox[0] == detection[0]
            ]

            num_gts = len(ground_truth_img)
            best_iou = 0

            for idx, gt in enumerate(ground_truth_img):
                # Calcula IoU entre detecção e ground truth
                iou = intersection_over_union(
                    torch.tensor(detection[3:]),
                    torch.tensor(gt[3:])
                )

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            if best_iou > iou_threshold:
                # Verifica se esta ground truth já foi detectada
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    TP[detection_idx] = 1
                    amount_bboxes[detection[0]][best_gt_idx] = 1
                else:
                    FP[detection_idx] = 1
            else:
                FP[detection_idx] = 1

        # Calcula precision e recall cumulativos
        TP_cumsum = torch.cumsum(TP, dim=0)
        FP_cumsum = torch.cumsum(FP, dim=0)
        recalls = TP_cumsum / (total_true_bboxes + epsilon)
        precisions = TP_cumsum / (TP_cumsum + FP_cumsum + epsilon)

        # Adiciona pontos (0,1) e (1,0) para interpolação
        precisions = torch.cat((torch.tensor([1]), precisions))
        recalls = torch.cat((torch.tensor([0]), recalls))

        # Calcula Average Precision usando integração trapezoidal
        average_precisions.append(torch.trapz(precisions, recalls))

    return sum(average_precisions) / len(average_precisions), average_precisions

print("✅ Função de cálculo de mAP implementada!")


✅ Função de cálculo de mAP implementada!


In [ ]:
# Importação necessária para Counter
from collections import Counter

# Função para avaliar modelo completo
def evaluate_model(model, test_loader, anchors, iou_threshold=0.5, conf_threshold=0.6):
    """
    Avalia o modelo no conjunto de teste e calcula métricas.

    Args:
        model: Modelo treinado
        test_loader: DataLoader de teste
        anchors: Âncoras do modelo
        iou_threshold: Limiar IoU para NMS
        conf_threshold: Limiar de confiança

    Returns:
        Dicionário com métricas de avaliação
    """
    print("🔍 Iniciando avaliação do modelo...")

    # Obtém predições e ground truth
    pred_boxes, true_boxes = get_evaluation_bboxes(
        test_loader, model, iou_threshold, anchors, conf_threshold
    )

    # Calcula mAP
    map_50, class_aps = calculate_map(pred_boxes, true_boxes, iou_threshold=0.5)
    map_75, _ = calculate_map(pred_boxes, true_boxes, iou_threshold=0.75)

    print(f"\n📊 RESULTADOS DA AVALIAÇÃO:")
    print(f"   mAP@0.5: {map_50:.4f}")
    print(f"   mAP@0.75: {map_75:.4f}")
    print(f"   Total de predições: {len(pred_boxes)}")
    print(f"   Total de ground truth: {len(true_boxes)}")

    # Mostra AP por classe
    print(f"\n📋 Average Precision por classe:")
    for i, ap in enumerate(class_aps):
        if i < len(PASCAL_CLASSES):
            print(f"   {PASCAL_CLASSES[i]}: {ap:.4f}")

    return {
        'mAP@0.5': map_50.item(),
        'mAP@0.75': map_75.item(),
        'class_aps': [ap.item() for ap in class_aps],
        'num_predictions': len(pred_boxes),
        'num_ground_truth': len(true_boxes)
    }

# Descomente para avaliar o modelo após treinamento
results = evaluate_model(model, test_loader, scaled_anchors)

print("✅ Função de avaliação completa implementada!")


✅ Função de avaliação completa implementada!


## 11. Visualização de Resultados

Implementamos funções para visualizar as predições do modelo e gerar exemplos de detecção.


In [ ]:
def show_predictions(model, dataloader, num_examples=4, conf_threshold=0.6, iou_threshold=0.5):
    """
    Mostra exemplos de predições do modelo.

    Args:
        model: Modelo treinado
        dataloader: DataLoader com imagens
        num_examples: Número de exemplos a mostrar
        conf_threshold: Limiar de confiança
        iou_threshold: Limiar IoU para NMS
    """
    model.eval()

    # Prepara âncoras
    anchors = (
        torch.tensor(ANCHORS) *
        torch.tensor(GRID_SIZES).unsqueeze(1).unsqueeze(1).repeat(1, 3, 2)
    ).to(device)

    examples_shown = 0

    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(dataloader):
            if examples_shown >= num_examples:
                break

            images = images.to(device)
            outputs = model(images)

            # Processa cada imagem no batch
            for img_idx in range(images.shape[0]):
                if examples_shown >= num_examples:
                    break

                # Coleta predições de todas as escalas
                bboxes = []

                for i in range(3):  # 3 escalas
                    batch_size, A, S, _, _ = outputs[i].shape
                    anchor = anchors[i]

                    boxes_scale_i = convert_cells_to_bboxes(
                        outputs[i], anchor, grid_size=S, is_predictions=True
                    )

                    bboxes += boxes_scale_i[img_idx]

                # Aplica NMS
                nms_boxes = non_max_suppression(
                    bboxes,
                    iou_threshold=iou_threshold,
                    confidence_threshold=conf_threshold
                )

                # Visualiza resultado
                print(f"\\n🖼️  Exemplo {examples_shown + 1} - {len(nms_boxes)} objetos detectados:")
                plot_image_with_boxes(images[img_idx], nms_boxes)

                examples_shown += 1

    model.train()

# Descomente para visualizar predições
show_predictions(model, test_loader, num_examples=3)

print("✅ Função de visualização de predições implementada!")
